# Improving Telescope Images with AI 🔭

_By: Gabriel Missael Barco, Nicolas Payot, Auriane Thilloy, Olivia Pereira, Noé Dia, Guillaume Payeur_

<a href="https://colab.research.google.com/github/GabrielMissael/super-resolution-workshop/blob/master/notebooks/Diffusion_Simulated_Galaxy_Pipeline_v2.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> <a href="https://github.com/GabrielMissael/super-resolution-workshop"><img src="https://img.shields.io/badge/View_on-GitHub-black?logo=github" alt="View on GitHub"/></a>

<img src="https://i.imgur.com/i8Rl4ef.png" alt="Introduction Banner" height="250"/>

**Welcome!** Today, we are going to mix **Astronomy** with **Artificial Intelligence**.

Real telescopes aren't perfect. The images they take are often blurry, pixelated, and noisy. In this workshop, we will learn how to fix them!

**The Plan:**
1. **Break it:** We will simulate a "bad" telescope to see how it ruins a perfect galaxy image.
2. **Fix it:** We will use a smart **AI model** that has learned what galaxies look like.
3. **Combine them:** We will teach the AI to look at a messy image and reconstruct the sharp galaxy hidden inside.
4. **The Mystery:** Can you identify a "Mystery Object" from a blurry blob? 👀
5. **The Contest:** Turn *yourself* (or any object) into a galaxy! The most creative images will win a prize! 🏆

## 1. Setting up the Laboratory 🤖

First, we need to load our tools. This code gets our "virtual telescope" ready and loads the **AI Brain** (a model that has studied thousands of galaxy images).

You don't need to understand the code in this specific cell; think of it as starting the engine of our car! 🚗

In [ ]:
# @title
print("Downloading data... 💾")
!git clone --quiet https://github.com/GabrielMissael/super-resolution-workshop

print("Getting some code... 🤖")
!pip3 install --quiet git+https://github.com/AlexandreAdam/score_models.git@dev

import sys
sys.path.append("super-resolution-workshop")
from src.diffusion_sampling.diffusion_sampling import *
from src.diffusion_sampling.telescope_app import launch_telescope_app
from pathlib import Path
from huggingface_hub import snapshot_download
from score_models import ScoreModel
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
import io
import gradio as gr
import numpy as np
import matplotlib.cm as cm

from huggingface_hub.utils import disable_progress_bars
disable_progress_bars()

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

MODEL_DIR = Path("model/galaxy_prior")

if not MODEL_DIR.exists():
    print("Downloading galaxy prior from Hugging Face... 🧐")
    snapshot_download(
        repo_id="GMissaelBarco/galaxy-prior",
        local_dir=MODEL_DIR,
        local_dir_use_symlinks=False,
        tqdm_class=None,
    )

model = ScoreModel(path=str(MODEL_DIR)).to(DEVICE)
model.load()
model.eval()
print("Model loaded ✅")

galaxies = torch.load("super-resolution-workshop/data/galaxies.pt", map_location=DEVICE)

show_grid(galaxies, title="Example clean galaxies")

%config InlineBackend.figure_format = 'retina'

## 2. Telescope Effect #1: The Blur (PSF) 👓

<img src="https://i.imgur.com/saE85st.png" alt="PSF example" height="150"/>

Real telescopes aren't perfect. Because of the atmosphere and the optics, light gets smeared out. Astronomers call this the **Point Spread Function (PSF)**, but you can just think of it as looking through **bad glasses**.

**Try it out:**
* Move the `Sigma PSF` slider.
* Watch how the details of the spiral arms melt away as the blur gets stronger.

In [ ]:
# @title
sigma_psf_slider_explore = widgets.FloatSlider(
    value=0.0,
    min=0.0,
    max=5.0,
    step=0.01,
    description="Sigma PSF:",
    layout=widgets.Layout(width="800px"),
)

out_psf = widgets.Output()

def update_psf_explore(change=None):
    with out_psf:
        clear_output(wait=True)
        psf_images = psf_on_image(galaxies, sigma=float(sigma_psf_slider_explore.value))
        show_grid(psf_images, title="Effect of PSF (blur only)")

sigma_psf_slider_explore.observe(update_psf_explore, names="value")
display(sigma_psf_slider_explore)
update_psf_explore()
display(out_psf)


## 3. Telescope Effect #2: Low Resolution 🧱

<img src="https://i.imgur.com/KC6LMZQ.png" alt="Downsampling example" height="150"/>

Digital cameras use **pixels**. If we don't have enough pixels, the image looks blocky (like in Minecraft or retro video games). We lose the tiny details.

**Try it out:**
* Lower the `Pixels downsampled to` slider.
* See how the galaxy turns into a grid of blocks.

In [ ]:
# @title
pixels_downsample_slider_explore = widgets.IntSlider(
    value=64,
    min=8,
    max=64,
    step=1,
    description="Pixels downsampled to:",
    layout=widgets.Layout(width="800px"),
)

out_downsample = widgets.Output()

def update_downsample_explore(change=None):
    with out_downsample:
        clear_output(wait=True)
        downsampled = downsample_img(galaxies, size=int(pixels_downsample_slider_explore.value))
        show_grid(downsampled, title="Effect of downsampling (resolution only)")

pixels_downsample_slider_explore.observe(update_downsample_explore, names="value")
display(pixels_downsample_slider_explore)
update_downsample_explore()
display(out_downsample)


## 4. Telescope Effect #3: The Noise (Static) 🌧️

<img src="https://i.imgur.com/ZBLnBF1.png" alt="Noise example" height="150"/>

Detectors aren't perfect either! Sometimes they record random signals, like "static" on an old TV or grain in a low-light photo.

**Try it out:**
* Increase the `Noise` slider.
* Notice how hard it becomes to see the galaxy shape when the "snow" takes over.

In [ ]:
# @title
sigma_noise_slider_explore = widgets.FloatSlider(
    value=0.00,
    min=0.0,
    max=0.5,
    step=0.0005,
    description="Noise σ:",
    layout=widgets.Layout(width="800px"),
)

out_noise = widgets.Output()

def update_noise_explore(change=None):
    with out_noise:
        clear_output(wait=True)
        noisy = add_gaussian_noise(galaxies, sigma=float(sigma_noise_slider_explore.value))
        show_grid(noisy, title="Effect of Gaussian noise")

sigma_noise_slider_explore.observe(update_noise_explore, names="value")
display(sigma_noise_slider_explore)
update_noise_explore()
display(out_noise)


## 5. Building a Realistic Telescope 🧪

<img src="https://i.imgur.com/tJAzpVK.png" alt="Pipeline example" height="150"/>

In the real world, **all three problems happen at once**. A telescope blurs the image, pixels make it blocky, and the camera adds noise.

**Your turn:**
1. Use the sliders below to ruin a perfect galaxy image.
2. **Pick a galaxy** you like best; we will try to save it in the next step!

Which parameter (Blur, Resolution, or Noise) makes the image hardest to recognize?

In [ ]:
# @title
# Shared sliders for the forward model used later in inference
sigma_psf_slider = widgets.FloatSlider(
    value=0.01,
    min=0.01,
    max=5.0,
    step=0.01,
    description="Sigma PSF:",
    layout=widgets.Layout(width="800px"),
)

pixels_downsample_slider = widgets.IntSlider(
    value=64,
    min=10,
    max=64,
    step=1,
    description="Pixels downsampled to:",
    layout=widgets.Layout(width="800px"),
)

sigma_noise_slider = widgets.FloatSlider(
    value=0.01,
    min=0.01,
    max=0.2,
    step=0.0005,
    description="Noise σ:",
    layout=widgets.Layout(width="800px"),
)

image_selector = widgets.ToggleButtons(
    options=[("Image 1", 0), ("Image 2", 1), ("Image 3", 2), ("Image 4", 3), ("Image 5", 4)],
    description="Select image:",
    layout=widgets.Layout(width="800px"),
)

out_pipeline = widgets.Output()

# Global variables to reuse in later cells
galaxies_psf = None
galaxies_downsampled = None
galaxies_noisy = None

def update_pipeline(change=None):
    global galaxies_psf, galaxies_downsampled, galaxies_noisy

    with out_pipeline:
        clear_output(wait=True)

        sigma_psf_val = float(sigma_psf_slider.value)
        size_val = int(pixels_downsample_slider.value)
        sigma_n_val = float(sigma_noise_slider.value)

        galaxies_psf = psf_on_image(galaxies, sigma=sigma_psf_val)
        galaxies_downsampled = downsample_img(galaxies_psf, size=size_val)
        galaxies_noisy = add_gaussian_noise(galaxies_downsampled, sigma_n_val)

        selected_idx = image_selector.value
        show_grid_final(galaxies_noisy, selected_idx=selected_idx)

for w in [sigma_psf_slider, pixels_downsample_slider, sigma_noise_slider, image_selector]:
    w.observe(update_pipeline, names="value")

ui = widgets.VBox(
    [
        sigma_psf_slider,
        pixels_downsample_slider,
        sigma_noise_slider,
        image_selector,
        out_pipeline,
    ]
)

display(ui)
update_pipeline()


## 6. The "Magic" Trick: Reconstructing the Galaxy 🔄

Here is the challenge: **Can we turn that ugly, noisy blob back into a sharp galaxy?**

To do this, we use a **Smart Guessing Game**:
1. **The Physics:** We know how the telescope broke the image (the blur and noise).
2. **The AI:** We have an AI that knows what *clean* galaxies usually look like.
3. **The Reconstruction:** The AI tries to draw a galaxy that fits the messy data we have, but also looks like a realistic galaxy.

We call these guesses **"Samples."** Let's see what the AI comes up with!

In [ ]:
# @title
# Build the linear operator A corresponding to the current PSF + downsampling
sigma_psf_val = float(sigma_psf_slider.value)
S_val = int(pixels_downsample_slider.value)
y_lin, A = psf_downsample_build_A(galaxies, sigma_psf=sigma_psf_val, S=S_val)


# Pick the selected galaxy and its noisy observation
idx = image_selector.value
y_obs = galaxies_noisy[idx]    # (S,S)
sigma_n = float(sigma_noise_slider.value)

sampler = LinearGaussianPosteriorSampler(
    observation=y_obs,   # (S,S)
    A=A,
    model=model,
    sigma_n=sigma_n,
    C=1.0,
    M=0.0,
)

samples = sampler.run(
    n_samples=4,
    steps=100,
    progress=True,
    true=galaxies[idx],       # for optional trajectory plotting
    plot_trajectory=True,    # set True if you want the animated view
    trajectory_stride=5,
)

print("Samples shape:", samples.shape)  # (4,1,Hs,Hs)


## 8. Did it work? 🎨

Let's look at the results!

* **Top Row (The Guesses):** The first image is the **True** answer. The next images are the AI's best guesses. Do they look similar?
* **Middle Row (The Check):** If we took the AI's guess and put it *back* through our bad telescope, does it match the data we observed?
* **Bottom Row (The Difference):** This shows what the AI missed. If this looks like random static (red and blue noise), the AI did a great job!

In [ ]:
# @title
fig, axes = plt.subplots(3, 5, figsize=(12, 6))

# Row 1: true + posterior samples
true_img = img_to_show(galaxies[idx], log_scale=True)
axes[0, 0].imshow(true_img, cmap="magma")
axes[0, 0].set_title("True")
axes[0, 0].axis("off")

for i in range(4):
    img = img_to_show(samples[i], log_scale=True)
    axes[0, i + 1].imshow(img, cmap="magma")
    axes[0, i + 1].set_title(f"Sample {i+1}")
    axes[0, i + 1].axis("off")

# Row 2: observed + mock observations
obs_img = img_to_show(y_obs, log_scale=False)
axes[1, 0].imshow(obs_img, cmap="magma")
axes[1, 0].set_title("Observed")
axes[1, 0].axis("off")

for i in range(4):
    mock = samples[i].view(1, sampler.Msrc) @ A.t()  # (1,Mobs)
    mock_img = mock.view(1, 1, y_obs.shape[0], y_obs.shape[1])
    mock_img_to_show = img_to_show(mock_img[0, 0], log_scale=False)
    axes[1, i + 1].imshow(mock_img_to_show, cmap="magma")
    axes[1, i + 1].set_title(f"Mock Obs {i+1}")
    axes[1, i + 1].axis("off")

    # Row 3: residuals
    residual = (y_obs - mock_img[0, 0]) / sigma_n
    residual_img = img_to_show(residual, log_scale=False)
    axes[2, i + 1].imshow(residual_img, cmap="bwr", vmin=-3, vmax=3)
    axes[2, i + 1].set_title(f"Residual {i+1}")
    axes[2, i + 1].axis("off")

axes[2, 0].axis("off")
plt.tight_layout()
plt.show()


## 9. The Best Guess & The Confusion Map 📊

Since the input was so messy, the AI might be unsure about some details.

* **Posterior Mean:** This is the average of all the AI's guesses. It's our single "best bet."
* **Uncertainty Map:** This lights up in the areas where the AI was confused or where the guesses varied a lot.

In [ ]:
# @title
samples_mean = samples.mean(dim=0)  # (1,Hs,Hs)
samples_std = samples.std(dim=0)    # (1,Hs,Hs)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

img_mean = img_to_show(samples_mean[0], log_scale=True)
axes[1].imshow(img_mean, cmap="magma")
axes[1].set_title("Posterior Mean")
axes[1].axis("off")

img_std = img_to_show(samples_std[0], log_scale=False)
axes[2].imshow(img_std, cmap="magma")
axes[2].set_title("Posterior Std Dev")
axes[2].axis("off")

true_img = img_to_show(galaxies[idx], log_scale=True)
axes[0].imshow(true_img, cmap="magma")
axes[0].set_title("True Image")
axes[0].axis("off")

plt.tight_layout()
plt.show()


## 10. The Mystery Galaxy Challenge 🕵️‍♀️

Okay, training is over. Now for the real mission.

Astronomers have captured 12 images of a **Mystery Object**. The telescope was small, the images are blurry, and we have no idea what it really is.

**Your Mission:** Use the pipeline we just built to clean up the data and reveal the hidden object. Ready?

### 10.1 The Data: 12 Noisy Exposures 📷

Below are the 12 images the telescope captured.

Right now, they just look like noisy blobs. But hidden inside is a very specific shape. Look closely at the grid—can you guess what is hiding behind the noise?

In [ ]:
# @title
sigma_n_ood = 0.1
sigma_psf_ood = 0.1
res_ood = 64

# Load observations of the mystery OOD galaxy
mistery_galaxy_obs = torch.load("super-resolution-workshop/data/mistery_galaxy_obs.pt")

sigma_n_ood = 0.1
sigma_psf_ood = 0.1
res_ood = 64

y_lin_ood, A = psf_downsample_build_A(
    mistery_galaxy_obs,
    sigma_psf=sigma_psf_ood,
    S=res_ood,
)

fig, ax = plt.subplots(2, 6, figsize=(12, 4))
for i in range(12):
    img = img_to_show(mistery_galaxy_obs[i], log_scale=False)
    ax[i // 6, i % 6].imshow(img, cmap="magma")
    ax[i // 6, i % 6].set_title(f"Observation {i+1}")
    ax[i // 6, i % 6].axis("off")
plt.tight_layout()
plt.show()


### 10.2 Asking the AI 🎲

Now we run the simulation!

1. We take the **12 noisy images**.
2. We use the **AI Brain** (Prior) to make sure the result looks real.
3. We generate **4 samples** (guesses) of what the true object is.

Let's see what it finds...

In [ ]:
# @title
sampler_mistery = LinearGaussianPosteriorSampler(
    observation=mistery_galaxy_obs,   # (B,res_ood,res_ood)
    A=A,
    model=model,
    sigma_n=sigma_n_ood,
    C=1.0,
    M=0.0,
)

samples_mistery = sampler_mistery.run(
    n_samples=4,
    steps=200,
    progress=True,
    true=None,
    plot_trajectory=True,
    trajectory_stride=5,
)

print("OOD samples shape:", samples_mistery.shape)  # (4,1,Hs,Hs)


### 10.3 Did we solve it? 👀

Before we look at the answer, let's check the math.

* **Top Row:** The AI's guesses (Samples).
* **Middle Row:** If we re-blurred those guesses, would they look like the noisy data?
* **Bottom Row:** The residuals (difference).

If the residuals look like random noise, our solution is scientifically valid!

In [ ]:
# @title
fig, axes = plt.subplots(3, 5, figsize=(12, 9))

# Top row: OOD posterior samples
axes[0, 0].axis("off")
for i in range(4):
    img = img_to_show(samples_mistery[i], log_scale=True, min_val=0.1)
    axes[0, i + 1].imshow(img, cmap="magma")
    axes[0, i + 1].set_title(f"Sample {i+1}")
    axes[0, i + 1].axis("off")

# Middle row: Mistery galaxy observations and mock observations
obs_img = img_to_show(mistery_galaxy_obs[0], log_scale=False)
axes[1, 0].imshow(obs_img, cmap="magma")
axes[1, 0].set_title("Observed")
axes[1, 0].axis("off")

for i in range(4):
    mock = samples_mistery[i].view(1, sampler_mistery.Msrc) @ A.t()  # (1,Mobs)
    mock_img = mock.view(1, 1, mistery_galaxy_obs.shape[1], mistery_galaxy_obs.shape[2])
    mock_img_to_show = img_to_show(mock_img[0, 0], log_scale=False)
    axes[1, i + 1].imshow(mock_img_to_show, cmap="magma")
    axes[1, i + 1].set_title(f"Mock Obs {i+1}")
    axes[1, i + 1].axis("off")

    residual = (mistery_galaxy_obs[0] - mock_img[0, 0]) / sigma_n_ood
    residual_img = img_to_show(residual, log_scale=False)
    axes[2, i + 1].imshow(residual_img, cmap="bwr", vmin=-3, vmax=3)
    axes[2, i + 1].set_title(f"Residual {i+1}")
    axes[2, i + 1].axis("off")

# Question mark for unknown true image
axes[0, 0].text(0.5, 0.5, "?", fontsize=40, ha="center", va="center")
axes[0, 0].set_title("True (unknown)")

axes[2, 0].axis("off")
plt.tight_layout()
plt.show()


### 10.4 The Big Reveal 🎉

Time to see the truth!

We finally load the **True Image**. Surprise! It wasn't a standard galaxy... it was a **Maple Leaf** 🇨🇦.

Look at how the AI managed to reconstruct the stem and the pointed tips, even though the data was just a fuzzy blob. This shows the power of combining Physics with AI!

In [ ]:
# @title
true_mistery = torch.load(
    "super-resolution-workshop/data/true_mistery_galaxy.pt"
).to(DEVICE)

# Plot posterior samples against the true mistery galaxy
fig, axes = plt.subplots(1, 5, figsize=(12, 3))
true_img = img_to_show(true_mistery[0], log_scale=True, min_val=0.1)
axes[0].imshow(true_img, cmap="magma")
axes[0].set_title("True Mistery Galaxy")
axes[0].axis("off")

for i in range(4):
    img = img_to_show(samples_mistery[i], log_scale=True, min_val=0.1)
    axes[i + 1].imshow(img, cmap="magma")
    axes[i + 1].set_title(f"Sample {i+1}")
    axes[i + 1].axis("off")
plt.tight_layout()
plt.show()

### 10.5 Final Comparison ✅

Here is the side-by-side:
1. **The True Object** (Maple Leaf).
2. **The AI's Best Guess** (Posterior Mean).
3. **What we actually saw** (The noisy observation).

From a grainy blob to a recognizable leaf using data + a learned prior. You’ve just used a state-of-the-art generative model to solve a real inverse problem! 🛰️🍁

In [ ]:
# @title
samples_mistery_mean = samples_mistery.mean(dim=0)  # (1,Hs,Hs)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

true_img = img_to_show(true_mistery, log_scale=True, min_val=0.2)
axes[0].imshow(true_img, cmap="magma")
axes[0].set_title("True Image")
axes[0].axis("off")

img_mean = img_to_show(samples_mistery_mean, log_scale=True, min_val=0.2)
axes[1].imshow(img_mean, cmap="magma")
axes[1].set_title("Posterior Mean")
axes[1].axis("off")

# Stacking result
axes[2].imshow(img_to_show(mistery_galaxy_obs[0], log_scale=False), cmap="magma")
axes[2].set_title("Observed (1st exposure)")
axes[2].axis("off")
plt.tight_layout()
plt.show()


# 🏆 The Final Challenge: Galaxy-fy Yourself! 🔭

Now, it's time for the **Creative Contest**!

**The Mission:**
1. **Take a photo** (of your face, a drawing, an object...) or upload one.
2. Use our "Fake Telescope" to ruin it with noise.
3. Use the AI to **reconstruct it as a galaxy**.

**🥇 How to Win:**
We will select the top images for a final vote. To win, your image must be **Creative** and hit the **Sweet Spot**:
* **Don't use too much noise:** If the AI *only* sees a galaxy and we can't recognize your original object, you won't win.
* **Don't use too little noise:** If it just looks like a regular photo with no "galaxy" style, you won't win.
* **Goal:** An image that looks like a cool galaxy *but* clearly resembles you or your object!

**📝 How to Submit & Vote:**
1. When you are happy with your result, **Right Click -> Save Image** to download it.
2. Click the **Padlet link** inside the app below.
3. Click the **(+)** button, write your **Name** as the title, and upload your image.
4. **Look at the board (or screen)** for the password—you will need it to enter the voting area and like your favorite images!
5. If you get stuck, wave your hand! Staff and professors are here to help.

**Ready? Run the code below and click the public link (ending in `gradio.live`) to play!**

In [ ]:
# @title
launch_telescope_app(
    model=model,
    LinearGaussianPosteriorSampler=LinearGaussianPosteriorSampler,
    psf_downsample_build_A=psf_downsample_build_A,
    share=True,   # good for Colab
    debug=False,
)
